In [1]:
import pandas as pd
from datasets import load_dataset

# Step 1: Download and load the EuroSAT dataset via Hugging Face
ds = load_dataset("tanganke/eurosat")

# Step 2: Extract target class names (10 land-use categories)
class_names = ds["train"].features["label"].names
print("EuroSAT Classes:", class_names)

# Step 3: Perform quick EDA to verify class balance
df_eda = ds["train"].to_pandas()
print("\nClass Distribution:")
print(df_eda["label"].value_counts().sort_index().rename(index=dict(enumerate(class_names))))

README.md:   0%|          | 0.00/5.01k [00:00<?, ?B/s]

c:\Users\anton\OneDrive\Documents\GitHub\adaptive-ml-platform\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anton\.cache\huggingface\hub\datasets--tanganke--eurosat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 73.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.13MB            

...

Generating jpeg_compression split:   0%|          | 0/2700 [00:00<?, ? examples/s]

Generating motion_blur split:   0%|          | 0/2700 [00:00<?, ? examples/s]

Generating pixelate split:   0%|          | 0/2700 [00:00<?, ? examples/s]

Generating spatter split:   0%|          | 0/2700 [00:00<?, ? examples/s]

EuroSAT Classes: ['annual crop land', 'forest', 'brushland or shrubland', 'highway or road', 'industrial buildings or commercial buildings', 'pasture land', 'permanent crop land', 'residential buildings or homes or apartments', 'river', 'lake or sea']

Class Distribution:
label
annual crop land                                2373
forest                                          2390
brushland or shrubland                          2381
highway or road                                 2012
industrial buildings or commercial buildings    2012
pasture land                                    1626
permanent crop land                             1995
residential buildings or homes or apartments    2431
river                                           1973
lake or sea                                     2407
Name: count, dtype: int64


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# Custom Dataset wrapper to convert Hugging Face PIL dictionary items into PyTorch Tensors
class EuroSATDataset(Dataset):
    def __init__(self, hf_ds, transform=None):
        self.ds = hf_ds
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        image = item["image"].convert("RGB") # Convert single/multi-channel to standard RGB
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

# Step 3: Minimal preprocessing & standard ImageNet normalization baseline
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Create PyTorch dataset instance
full_ds = EuroSATDataset(ds["train"], transform=transform)

# Split 80% training / 20% validation with fixed random seed
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(
    full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

# Initialize DataLoaders for batching
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_ds)} | Validation samples: {len(val_ds)}")

Train samples: 17280 | Validation samples: 4320


In [3]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Auto-select CUDA / MPS / CPU execution device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Step 4: Load pretrained MobileNetV3-Small weights & adjust final layer for 10 classes
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10)
model = model.to(device)

# Set optimization criterion and fine-tuning learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Training loop over 3 epochs
epochs = 3
model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * x.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Training Loss: {epoch_loss:.4f}")

Training on device: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\anton/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 12.6MB/s]


Epoch 1/3 - Training Loss: 1.0487
Epoch 2/3 - Training Loss: 0.3922
Epoch 3/3 - Training Loss: 0.2698


In [4]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Step 5: Run evaluation inference loop on validation set
model.eval()
preds, y_val = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        logits = model(x)
        predictions = torch.argmax(logits, dim=1)
        preds.extend(predictions.cpu().numpy())
        y_val.extend(y.numpy())

# Compute evaluation summary metrics
accuracy = np.mean(np.array(preds) == np.array(y_val))
macro_f1 = f1_score(y_val, preds, average="macro")

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Macro F1 Score:     {macro_f1:.4f}\n")
print("Detailed Classification Report:\n")
print(classification_report(y_val, preds, target_names=class_names))
print("Confusion Matrix:\n")
print(confusion_matrix(y_val, preds))

Validation Accuracy: 0.9208
Macro F1 Score:     0.9164

Detailed Classification Report:

                                              precision    recall  f1-score   support

                            annual crop land       0.96      0.89      0.92       491
                                      forest       0.99      0.97      0.98       499
                      brushland or shrubland       0.88      0.87      0.87       464
                             highway or road       0.87      0.85      0.86       399
industrial buildings or commercial buildings       0.94      0.96      0.95       382
                                pasture land       0.86      0.92      0.89       318
                         permanent crop land       0.78      0.90      0.84       377
residential buildings or homes or apartments       0.98      0.97      0.98       514
                                       river       0.93      0.88      0.91       366
                                 lake or sea      

In [5]:
import os
import json

# Ensure target storage folder exists at root level relative to notebooks dir
os.makedirs("../models/satellite", exist_ok=True)

# Step 6: Save fine-tuned PyTorch state dictionary
torch.save(model.state_dict(), "../models/satellite/model_v1.pt")

# Export JSON metrics artifact for model registry integration
with open("../models/satellite/metrics_v1.json", "w") as f:
    json.dump({
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1)
    }, f, indent=2)

# Step 7: Extract penultimate layer embeddings (global average pooling) for Phase 6 drift engine
embeddings = []
model.eval()

with torch.no_grad():
    sample_count = 0
    for x, _ in val_loader:
        x = x.to(device)
        # Extract features right before classification head
        feat = model.features(x)
        pooled = feat.mean(dim=[2, 3])  # Spatial Global Average Pooling [B, C, H, W] -> [B, C]
        embeddings.append(pooled.cpu())
        
        sample_count += x.size(0)
        if sample_count >= 500: # Retain a reference slice of 500 embeddings
            break

# Concatenate into single reference tensor and export to disk
reference_embeddings = torch.cat(embeddings)[:500]
torch.save(reference_embeddings, "../models/satellite/reference_embeddings_v1.pt")

print("Successfully exported model, metrics, and baseline embeddings to models/satellite/")
print(f"Reference Embeddings Shape: {reference_embeddings.shape}")

Successfully exported model, metrics, and baseline embeddings to models/satellite/
Reference Embeddings Shape: torch.Size([500, 576])
